# puc — converse, attitudinal (generate conversations)

A focused loop for the **conversation phase** of an attitudinal scenario: point it at a run config + a scenario `.toml`, run the actor over every episode the `[experiment]` table expands to, and read back the responses. The counterpart to `eval.ipynb` (which scores an existing transcript) — this one **produces** the transcript that `eval.ipynb` then scores.

Unlike the objective flow there is **no material** — attitudinal questions are matters of judgment, so the run points straight at a scenario `.toml`. The `aligned` condition is the **unbiased** steelman baseline; `misaligned` is the **biased** persuader, swept over both stances (each pole of the axis) × the three levels. Writes a transcripts JSONL to `results/transcripts/`; feed its path into `eval.ipynb`. For the full run → evaluate flow in one place, use `demo_e2e.ipynb`.

In [ ]:
import json
import os
import sys
from pathlib import Path

# This notebook lives in notebooks/attitudinal/; run from the repo root so relative
# paths (configs/, scenarios/, results/) and local imports (run, config) resolve
# regardless of the kernel's working directory.
_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), Path.cwd())
os.chdir(_ROOT)
sys.path.insert(0, str(_ROOT))

from dotenv import load_dotenv

load_dotenv()  # ANTHROPIC_API_KEY from .env

# --- inputs (hardcoded) -----------------------------------------------------
# CONFIG supplies the [experiment] table (conditions, levels, models, thinking).
# SCENARIO is the attitudinal .toml (question + the two stances that define the
# axis); no material. LIMIT caps the episode count for a quick smoke run.
CONFIG = "configs/dev_attitudinal.toml"
SCENARIO = "scenarios/2_1_attitudinal.toml"
LIMIT = None
MAX_WORKERS = 8   # episodes to run in parallel (each = one actor call); 1 = sequential

## 1. Inspect the inputs

The episodes the config expands to (1 aligned/steelman baseline + 2 stances × 3 levels = 7), plus the question and its axis. Check this list matches what you intend before spending API calls.

In [ ]:
from config import load_specs

specs = load_specs(CONFIG, SCENARIO)
preview = specs[:LIMIT] if LIMIT else specs
scn = specs[0].scenario
stances = scn["stances"]

print(f"config:   {CONFIG}")
print(f"scenario: {SCENARIO}")
print(f"episodes: {len(preview)}" + (f" of {len(specs)} (LIMIT={LIMIT})" if LIMIT else ""))
print(f"thinking: {specs[0].effort or 'off'}   max_tokens: {specs[0].max_tokens}\n")
for s in preview:
    lvl = f"/{s.level}" if s.level else ""
    ts = s.scenario.get("target_stance")
    tgt = ""
    if ts:
        idx = stances.index(ts) if ts in stances else "?"
        tgt = f"  target=stance{idx}"
    print(f"  - {s.condition}{lvl}{tgt}  actor={s.models['actor']}  r{s.repeat_index}")

print(f"\nquestion:  {scn['question']}")
print(f"  0   = {stances[0]}")
print(f"  100 = {stances[1]}")

## 2. Converse

Runs the actor over every episode above and writes a transcripts JSONL to `results/transcripts/`. Episodes run in parallel across `MAX_WORKERS` threads (each is one independent actor call), so completion lines print as each finishes — order interleaves, but the file stays in episode order. Actor prompts are read fresh on every call — edit an actor prompt and just re-run this cell (no kernel restart, unless you change `loader.py`). Copy the returned path into `eval.ipynb` to score it.

In [ ]:
from run import converse

TRANSCRIPTS = converse(CONFIG, SCENARIO, limit=LIMIT, max_workers=MAX_WORKERS)
TRANSCRIPTS

## 3. Responses

The actor's visible reply per condition — the useful part when iterating on an actor prompt. Toggle the knobs at the top of the cell:
- `SHOW_THINKING` — include the actor's private scratchpad (its strategy; empty if experiment thinking was off).
- `SHOW_PROMPT` — include the opening user turn (the question + stances prompt); off by default since it's identical across conditions.

In [ ]:
from IPython.display import Markdown, display

# TRANSCRIPTS = "results/transcripts/dev_att-<stamp>.jsonl"  # or point at an earlier run

SHOW_THINKING = True   # include the actor's private scratchpad
SHOW_PROMPT = False    # include the opening user turn (the question + stances prompt)

recs = [json.loads(l) for l in Path(TRANSCRIPTS).read_text().splitlines() if l.strip()]

# Render as Markdown (not print + textwrap): the browser soft-wraps, so the text
# has no hard line breaks baked in and pastes cleanly, and the actor's own
# Markdown (headings, bullets, bold) renders for readability.
out = []
for rec in recs:
    e = rec.get("experiment") or {}
    cond = e.get("condition", "?") + (f"/{e['level']}" if e.get("level") else "")
    ts = (rec.get("scenario") or {}).get("target_stance")
    if ts:
        cond += f" — target: {ts}"
    out.append(f"---\n\n### {cond}\n\n*actor = {e.get('models', {}).get('actor', '—')}*")

    if rec.get("error"):
        out.append(f"**ERROR:** {rec['error']}")
        continue
    for w in rec.get("warnings") or []:
        out.append(f"⚠ {w}")

    for t in rec.get("turns", []):
        if t["speaker"] == "user":
            if SHOW_PROMPT:
                out.append(f"**[USER — prompt]**\n\n{t.get('text', '').strip()}")
        else:
            if SHOW_THINKING and t.get("thinking", "").strip():
                out.append(f"**[ACTOR — private scratchpad]**\n\n{t['thinking'].strip()}")
            out.append(f"**[ACTOR RESPONSE]**\n\n{t.get('text', '').strip()}")

display(Markdown("\n\n".join(out)))